In [10]:
import pandas as pd
import requests
import time
import json
from Bio import Entrez, Medline
from tqdm import tqdm

In [24]:
# Setup email and optional API key
Entrez.email = "LING.THANG@CUANSCHUTZ.EDU"
Entrez.api_key = "07db9c0ae477cc1432e0c6a1d024d64b0c09"

# --- Search Parameters ---
# How many articles to retrieve per drug.
MAX_RESULTS_PER_DRUG = 1000
# Base query terms for PubMed.
DISEASE_TERMS = "Mycobacterium tuberculosis"
HDT_TERMS = '"host-directed therapy" OR "HDT"'


# Literature Searcher Class

In [16]:

class LiteratureSearcher:
    """
    A class to find literature evidence for drug candidates by searching
    PubMed directly and mechanistically via DGIdb.
    """
    def __init__(self, email, api_key=None):
        Entrez.email = email
        if api_key:
            Entrez.api_key = api_key
        self.dgidb_url = "https://dgidb.org/api/v2/interactions.json"

    def _query_dgidb(self, drug_name):
        """Queries DGIdb to find gene targets for a given drug."""
        params = {'drugs': drug_name, 'interaction_sources': 'DrugBank'}
        try:
            response = requests.get(self.dgidb_url, params=params)
            response.raise_for_status()
            data = response.json()
            # Extract unique gene names from the interactions
            genes = {item['geneName'] for item in data.get('matchedTerms', [])[0].get('interactions', [])}
            return list(genes)
        except requests.exceptions.RequestException as e:
            print(f"  - DGIdb query failed for '{drug_name}': {e}")
            return []

    def _search_pubmed(self, query):
        """Performs a search on PubMed and returns the count and PMIDs."""
        try:
            handle = Entrez.esearch(db="pubmed", term=query, retmax=MAX_RESULTS_PER_DRUG)
            record = Entrez.read(handle)
            handle.close()
            count = int(record.get('Count', 0))
            pmids = record.get('IdList', [])
            return count, pmids
        except Exception as e:
            print(f"  - PubMed esearch failed for query '{query[:50]}...': {e}")
            return 0, []

    def _fetch_pubmed_summaries(self, pmids):
        """Fetches structured summaries for a list of PMIDs using Medline parsing."""
        if not pmids:
            return []
        
        summaries = []
        try:
            handle = Entrez.efetch(db="pubmed", id=",".join(pmids), rettype="medline", retmode="text")
            # Use Medline.parse for MEDLINE formatted text records
            records = Medline.parse(handle)
            
            for record in records:
                # Extract details safely using .get() with default values
                title = record.get("TI", "No Title Found")
                authors = record.get("AU", [])
                pmid = record.get("PMID", "")
                
                # DOI is often in the 'LID' or 'AID' field for articles from PubMed Central
                doi = ""
                # Check 'LID' first, as it's a common place for the DOI
                if "LID" in record and "[doi]" in record["LID"]:
                    doi = record["LID"].split(" ")[0]
                else:
                    # Fallback to checking 'AID'
                    for aid in record.get("AID", []):
                        if "[doi]" in aid:
                            doi = aid.split(" ")[0]
                            break
                        
                summaries.append({
                    "title": title,
                    "authors": ", ".join(authors),
                    "pmid": pmid,
                    "doi": doi
                })
            handle.close()
            return summaries
        except Exception as e:
            print(f"  - PubMed efetch failed for PMIDs {pmids}: {e}")
            return []

    def find_evidence(self, drug_name):
        """
        Main search function. Implements a tiered search strategy.
        """
        print(f"\nSearching for '{drug_name}'...")
        
        # --- Tier 1: Direct Search ---
        print("  - Performing direct search (including HDT terms)...")
        # First, try a very specific query including HDT terms
        specific_direct_query = f'("{drug_name}"[Title/Abstract]) AND ({DISEASE_TERMS}) AND ({HDT_TERMS})'
        count, pmids = self._search_pubmed(specific_direct_query)
        time.sleep(0.34)

        if pmids:
            print(f"  - Found {count} direct results with HDT terms. Fetching details...")
            summaries = self._fetch_pubmed_summaries(pmids)
            return "Direct (HDT)", count, summaries

        # If that fails, fall back to a broader direct search
        print("  - No direct HDT results found. Performing broader direct search...")
        broad_direct_query = f'("{drug_name}"[Title/Abstract]) AND ({DISEASE_TERMS})'
        count, pmids = self._search_pubmed(broad_direct_query)
        time.sleep(0.34)

        if pmids:
            print(f"  - Found {count} direct results. Fetching details...")
            summaries = self._fetch_pubmed_summaries(pmids)
            return "Direct (TB)", count, summaries

        # --- Tier 2: Mechanistic Search (if Tier 1 fails) ---
        print("  - No direct results found. Performing mechanistic search via DGIdb...")
        targets = self._query_dgidb(drug_name)
        time.sleep(0.34)

        if not targets:
            print("  - No gene targets found in DGIdb.")
            return "No Results", 0, []
            
        print(f"  - Found targets in DGIdb: {', '.join(targets[:3])}...")
        target_query_part = " OR ".join([f'"{target}"[Title/Abstract]' for target in targets])
        
        # Try specific mechanistic search first
        specific_mechanistic_query = f'({target_query_part}) AND ({DISEASE_TERMS}) AND ({HDT_TERMS})'
        count, pmids = self._search_pubmed(specific_mechanistic_query)
        time.sleep(0.34)
        
        if pmids:
            print(f"  - Found {count} mechanistic results with HDT terms. Fetching details...")
            summaries = self._fetch_pubmed_summaries(pmids)
            return "Mechanistic (HDT)", count, summaries

        # Fallback to broad mechanistic search
        print("  - No mechanistic HDT results found. Performing broader mechanistic search...")
        broad_mechanistic_query = f'({target_query_part}) AND ({DISEASE_TERMS})'
        count, pmids = self._search_pubmed(broad_mechanistic_query)
        time.sleep(0.34)

        if pmids:
            print(f"  - Found {count} mechanistic results. Fetching details...")
            summaries = self._fetch_pubmed_summaries(pmids)
            return "Mechanistic (TB)", count, summaries

        print("  - No mechanistic results found.")
        return "No Results", 0, []


# Output Formatting

In [17]:
def format_evidence_for_tsv(summaries):
    """
    Formats the list of summaries into a clean, multi-line string for TSV output.
    """
    if not summaries or not isinstance(summaries, list):
        return "No evidence found"
        
    formatted_strings = []
    for i, s in enumerate(summaries):
        # Create a clean, readable entry for each paper
        entry = (
            f"[{i+1}] {s.get('title', 'N/A')}\n"
            f"    Authors: {s.get('authors', 'N/A')}\n"
            f"    PMID: {s.get('pmid', 'N/A')} | DOI: {s.get('doi', 'N/A')}"
        )
        formatted_strings.append(entry)
        
    return "\n\n".join(formatted_strings)


# Main Execution

In [28]:

if __name__ == "__main__":
    # --- Load Top 20 Drugs ---
    try:
        drug_pred_results = pd.read_csv("../results/final_high_confidence_drug_predictions.tsv", sep="\t")
        top_drugs = drug_pred_results.nlargest(20, 'combined_z_score').copy()
        drug_names = top_drugs['drug_name'].unique()
    except FileNotFoundError:
        print("Error: Input file not found. Please check the path.")
        exit()

    # --- Initialize Search and Output ---
    searcher = LiteratureSearcher(email=Entrez.email, api_key=Entrez.api_key)
    # This list will store dictionaries, which is easier to work with
    results_list = []

    # --- Main Loop ---
    for drug in tqdm(drug_names, desc="Processing Drugs"):
        search_type, count, summaries = searcher.find_evidence(drug)
        
        # Get the original combined_z_score for the drug
        z_score = top_drugs.loc[top_drugs['drug_name'] == drug, 'combined_z_score'].iloc[0]

        results_list.append({
            "drug_name": drug,
            "combined_z_score": z_score,
            "search_type": search_type,
            "studies_found": count,
            "literature_evidence": summaries # Store the structured data
        })

    # --- Create Final DataFrame and Save ---
    final_df = pd.DataFrame(results_list)
    
    # Create a version for saving to TSV with formatted evidence
    df_for_tsv = final_df.copy()
    df_for_tsv['literature_evidence'] = df_for_tsv['literature_evidence'].apply(format_evidence_for_tsv)
    
    # Save to TSV
    output_path_tsv = "../results/drug_predictions_with_literature.tsv"
    df_for_tsv.to_csv(output_path_tsv, sep="\t", index=False)
    print(f"\n✅ Literature search complete. Human-readable output saved to: {output_path_tsv}")

    # Save to JSON for better data structure preservation
    output_path_json = "../results/drug_predictions_with_literature.json"
    final_df.to_json(output_path_json, orient="records", indent=4)
    print(f"✅ Structured JSON output saved to: {output_path_json}")

Processing Drugs:   0%|          | 0/20 [00:00<?, ?it/s]


Searching for 'nilotinib'...
  - Performing direct search (including HDT terms)...
  - Found 1 direct results with HDT terms. Fetching details...


Processing Drugs:   5%|▌         | 1/20 [00:00<00:17,  1.08it/s]


Searching for 'mycophenolic-acid'...
  - Performing direct search (including HDT terms)...
  - No direct HDT results found. Performing broader direct search...
  - Found 3 direct results. Fetching details...


Processing Drugs:  10%|█         | 2/20 [00:02<00:23,  1.32s/it]

...


Searching for 'loperamide'...
  - Performing direct search (including HDT terms)...
  - No direct HDT results found. Performing broader direct search...
  - Found 4 direct results. Fetching details...


Processing Drugs:  95%|█████████▌| 19/20 [00:31<00:01,  1.79s/it]


Searching for 'paclitaxel'...
  - Performing direct search (including HDT terms)...
  - No direct HDT results found. Performing broader direct search...
  - Found 1 direct results. Fetching details...


Processing Drugs: 100%|██████████| 20/20 [00:34<00:00,  1.70s/it]


✅ Literature search complete. Human-readable output saved to: ../results/drug_predictions_with_literature.tsv
✅ Structured JSON output saved to: ../results/drug_predictions_with_literature.json


In [27]:
tuberculosis_Mycobacterium_tuberculosis

,drug_name,combined_z_score,search_type,studies_found,literature_evidence
0,nilotinib,2.524399,Direct (HDT),1,[1] Mycobacterium tuberculosis resides in lyso...
1,mycophenolic-acid,2.495033,Direct (TB),11,[1] A new target for drug repositioning: CEBPa...
2,resveratrol,1.976094,Direct (HDT),1,[1] Sirtuin 3 is essential for host defense ag...
3,cilastatin,1.721115,Direct (TB),23,[1] A rare case of multiple brain abscesses ca...
4,menadione,1.697798,Direct (TB),21,"[1] Design, Synthesis, and In Vitro Evaluation..."
5,rosuvastatin,1.627595,Direct (TB),4,[1] PET-CT outcomes from a randomised controll...
6,fludroxycortide,1.604148,No Results,0,No evidence found
7,lapatinib,1.590347,Direct (TB),1,[1] Identification of the novel markers of PPA...
8,fostamatinib,1.573205,Direct (TB),1,[1] Inhibition of spleen tyrosine kinase in th...
9,fenbendazole,1.503700,No Results,0,No evidence found


In [29]:
df_for_tsv

,drug_name,combined_z_score,search_type,studies_found,literature_evidence
0,nilotinib,2.524399,Direct (HDT),1,[1] Mycobacterium tuberculosis resides in lyso...
1,mycophenolic-acid,2.495033,Direct (TB),3,[1] A new target for drug repositioning: CEBPa...
2,resveratrol,1.976094,Direct (HDT),1,[1] Sirtuin 3 is essential for host defense ag...
3,cilastatin,1.721115,Direct (TB),17,[1] A rare case of multiple brain abscesses ca...
4,menadione,1.697798,Direct (TB),18,"[1] Design, Synthesis, and In Vitro Evaluation..."
5,rosuvastatin,1.627595,No Results,0,No evidence found
6,fludroxycortide,1.604148,No Results,0,No evidence found
7,lapatinib,1.590347,No Results,0,No evidence found
8,fostamatinib,1.573205,No Results,0,No evidence found
9,fenbendazole,1.503700,No Results,0,No evidence found


In [30]:
df_for_tsv["drug_name"]

0             nilotinib
1     mycophenolic-acid
2           resveratrol
3            cilastatin
4             menadione
5          rosuvastatin
6       fludroxycortide
7             lapatinib
8          fostamatinib
9          fenbendazole
10            minoxidil
11            verapamil
12          simvastatin
13            sirolimus
14            terguride
15            topotecan
16             dopamine
17          chloroquine
18           loperamide
19           paclitaxel
Name: drug_name, dtype: object

In [33]:
# save to TSV
df_for_tsv[["drug_name", "combined_z_score"]].to_csv("../results/drugnames.tsv", sep="\t", index=False)